# Phi-4-mini support router: train or recover, then measure
Use a GPU runtime. Every prediction is generated by the HF model; there are no mock responses or prefilled scores.
Hyperparameters, model identity and dependency pins are explicit configuration, not results.
**For your existing trained adapter, choose `recover` below. You do not need to retrain.**
Use the newly packaged source ZIP with this notebook. Each run creates a new directory and preserves previous runs.
Do not upgrade packages manually between cells. All ML commands use one isolated Python environment, not the Colab kernel.
If any cell fails, stop and download diagnostics using the evidence cell near the end. A passing training check is not proof of useful accuracy.


In [ ]:
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import os, sys, subprocess, zipfile, json, shutil

RUN_MODE = input('Mode [recover/train]: ').strip().lower()
if RUN_MODE not in {'recover', 'train'}:
    raise ValueError('Enter recover or train')
print('Upload the NEW support-ticket-router-source.zip')
uploads = files.upload()
archives = [name for name in uploads if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError('Upload exactly one source ZIP')
root = Path('/content') / ('support-router-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
root.mkdir()
def extract_checked(archive, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(archive) as z:
        for entry in z.infolist():
            if not (destination / entry.filename).resolve().is_relative_to(destination):
                raise ValueError('Unsafe ZIP path')
        z.extractall(destination)
extract_checked(archives[0], root)
os.chdir(root)
if not Path('scripts/workflow.py').is_file():
    raise ValueError('This is the old source ZIP; upload the newly generated ZIP')
Path('artifacts').mkdir()
Path('artifacts/run-mode.txt').write_text(RUN_MODE)
del uploads

def run(*args):
    # Stream and retain full subprocess output, including chained tracebacks.
    with Path('artifacts/execution.log').open('a', encoding='utf-8') as log:
        log.write('COMMAND: ' + repr(args) + '\n')
        with subprocess.Popen([str(x) for x in args], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
            status = proc.wait()
        if status:
            raise subprocess.CalledProcessError(status, args)

print('Run directory:', root)


## Recovery inputs (skipped for a new training run)
Upload `phi4-router-adapter.zip` and `week5-evidence.zip` from the previous run.
Only the adapter is reused; the old merged model and failed scores are not used as new results.
The old manifest and base revision are retained to ensure the same data and checkpoint are used.


In [ ]:
if RUN_MODE == 'recover':
    print('Upload the previous adapter ZIP and evidence ZIP together')
    uploads = files.upload()
    adapter_archive = evidence_archive = None
    for name in uploads:
        if not name.endswith('.zip'):
            continue
        with zipfile.ZipFile(name) as z:
            if 'adapter_model.safetensors' in z.namelist():
                adapter_archive = name
            if 'data/prepared/manifest.json' in z.namelist():
                evidence_archive = name
    if not adapter_archive or not evidence_archive:
        raise ValueError('Both original archives are required for verified recovery')
    extract_checked(adapter_archive, 'artifacts/phi4-router-adapter')
    with zipfile.ZipFile(evidence_archive) as z:
        provenance = json.loads(z.read('artifacts/model-provenance.json'))
        previous_manifest = json.loads(z.read('data/prepared/manifest.json'))
        previous_commit = z.read('artifacts/llamafactory-commit.txt').decode().strip()
    Path('artifacts/recovery-provenance.json').write_text(json.dumps(provenance, indent=2))
    Path('artifacts/recovery-manifest.json').write_text(json.dumps(previous_manifest, indent=2))
    Path('artifacts/recovery-factory-commit.txt').write_text(previous_commit)
    del uploads
    print('Adapter restored; new evaluation outputs will be generated.')


## Install an isolated environment
The default factory commit comes from the recorded completed training run. The core dependency versions are in `configs/colab-constraints.txt`.
This avoids Colab's unrelated preinstalled package conflicts and preserves one Transformers version throughout.
Installation requires network access and disk space; a fresh GPU run is still required to validate the complete workflow.


In [ ]:
FACTORY_REVISION = '702005ce5237ae10f3c572c3f7741ecfd1422fc9'
if RUN_MODE == 'recover':
    FACTORY_REVISION = Path('artifacts/recovery-factory-commit.txt').read_text().strip()
factory = root / 'LLaMA-Factory'
venv = root / '.venv'
run(sys.executable, '-m', 'pip', 'install', 'virtualenv')
run(sys.executable, '-m', 'virtualenv', '--python', sys.executable, str(venv))
PYTHON = str(venv / 'bin/python')
CLI = str(venv / 'bin/llamafactory-cli')
# Prevent user-installed packages or external PYTHONPATH from leaking into children.
os.environ['PYTHONNOUSERSITE'] = '1'
os.environ.pop('PYTHONPATH', None)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
run('git', 'clone', 'https://github.com/hiyouga/LLaMA-Factory.git', str(factory))
run('git', '-C', str(factory), 'checkout', '--detach', FACTORY_REVISION)
run(PYTHON, '-m', 'pip', 'install', '-c', 'configs/colab-constraints.txt',
    '-e', str(factory), '-e', '.[inference]', 'bitsandbytes')
run(PYTHON, '-m', 'pip', 'check')
Path('artifacts/llamafactory-commit.txt').write_text(
    subprocess.check_output(['git', '-C', str(factory), 'rev-parse', 'HEAD'], text=True))
run('nvidia-smi')
run(PYTHON, '-m', 'scripts.workflow', 'environment', '--stage', 'initial')

def python_cell(source):
    # Save exactly the code executed in a fresh process for later review.
    folder = Path('artifacts/executed-cells')
    folder.mkdir(exist_ok=True)
    script = folder / ('cell-' + str(len(list(folder.glob('*.py')))) + '.py')
    script.write_text(source, encoding='utf-8')
    run(PYTHON, str(script.resolve()))


## Download and prepare the real dataset
Choose columns from the actual CSV headers. Recovery must reproduce the original manifest exactly; a mismatch stops the run.
Training and validation labels are targets for scoring, never substituted for generated predictions.


In [ ]:
import urllib.request, csv
raw = Path('data/raw/support_tickets.csv')
raw.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/The-Gen-Academy/5A-Fine-Tune-a-Support-Ticket-Router/main/support_tickets.csv', raw)
with raw.open(encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    print('Columns:', reader.fieldnames)
    print('Rows:', sum(1 for _ in reader))
TEXT_COLUMN = input('Ticket text column: ').strip()
LABEL_COLUMN = input('Routing label column: ').strip()
run(PYTHON, '-m', 'ticket_router.prepare', str(raw), '--text-column', TEXT_COLUMN, '--label-column', LABEL_COLUMN)
if RUN_MODE == 'recover':
    actual = json.loads(Path('data/prepared/manifest.json').read_text())
    expected = json.loads(Path('artifacts/recovery-manifest.json').read_text())
    if actual != expected:
        raise ValueError('Dataset differs from original training run; restore the original CSV before proceeding')


## Resolve checkpoint and run preflight
Recovery downloads the exact base revision recorded in the original evidence. New training records the revision it downloads.
Preflight checks real token lengths, split integrity, template parity and a CUDA 4-bit operation.


In [ ]:
python_cell("""
from pathlib import Path
import json, yaml
from huggingface_hub import snapshot_download
from ticket_router.domain import BASE_MODEL
recovery = Path('artifacts/recovery-provenance.json')
previous = json.loads(recovery.read_text()) if recovery.exists() else None
if previous and previous['model'] != BASE_MODEL:
    raise ValueError('Recovery checkpoint does not match the selected model')
snapshot = snapshot_download(BASE_MODEL, revision=previous['revision'] if previous else None)
Path('artifacts/base-snapshot-path.txt').write_text(snapshot)
Path('artifacts/model-provenance.json').write_text(json.dumps(
    {'model': BASE_MODEL, 'snapshot': snapshot, 'revision': Path(snapshot).name}, indent=2))
for source, target in [('configs/phi4_qlora.yaml', 'artifacts/train-resolved.yaml'),
                       ('configs/merge.yaml', 'artifacts/merge-resolved.yaml')]:
    config = yaml.safe_load(Path(source).read_text())
    config['model_name_or_path'] = snapshot
    Path(target).write_text(yaml.safe_dump(config))
""")
run(PYTHON, '-m', 'scripts.preflight')


## Train only if requested, then verify the adapter
Recovery verifies the saved training state and actual adapter tensors without repeating training.
A new run must finish all configured steps and produce finite, learned adapter weights.


In [ ]:
run(PYTHON, '-m', 'scripts.workflow', 'environment', '--stage', 'before-training')
if RUN_MODE == 'train':
    run(CLI, 'train', 'artifacts/train-resolved.yaml')
run(PYTHON, '-m', 'scripts.workflow', 'verify-training')
from IPython.display import display, Image
for plot in Path('artifacts/phi4-router-adapter').glob('*.png'):
    display(Image(filename=str(plot)))
# Preserve the adapter immediately, even if later export/inference fails.
shutil.make_archive(str(root / 'phi4-router-adapter'), 'zip', 'artifacts/phi4-router-adapter')
files.download(str(root / 'phi4-router-adapter.zip'))


## Export under the same environment and verify the merged files
Export uses CPU RAM and writes a new merged directory. The old Transformers-5 export is not reused.
Verification checks actual weight shards, finite tensors and identical baseline/merged prompt tokens.


In [ ]:
run(PYTHON, '-m', 'scripts.workflow', 'environment', '--stage', 'before-merge')
run(CLI, 'export', 'artifacts/merge-resolved.yaml')
run(PYTHON, '-m', 'scripts.workflow', 'verify-merge')
run(PYTHON, '-m', 'scripts.workflow', 'environment', '--stage', 'after-merge')


## Real smoke inference
Uses one actual held-out ticket from each class, loaded from the prepared dataset. The expected label is used only for scoring.
Full tracebacks stop execution. If every output is invalid, stop to investigate. Valid but wrong predictions are retained honestly.
These cases are a subset of validation, not an independent benchmark. Do not tune against them.


In [ ]:
run(PYTHON, '-m', 'scripts.workflow', 'smoke')


## Evaluate the matched baseline and candidate
Both use the same prompt, FP16 GPU inference, deterministic decoding and cache-disabled generation.
Each command starts a fresh process and releases its model on exit. Any inference exception stops with a full traceback instead of producing a false completed score report.
Invalid model text still counts as incorrect. No scores or speed improvement are predetermined.


In [ ]:
run(PYTHON, '-m', 'scripts.workflow', 'environment', '--stage', 'before-evaluation')
snapshot = Path('artifacts/base-snapshot-path.txt').read_text().strip()
run(PYTHON, '-m', 'ticket_router.evaluate', 'run', '--model', snapshot, '--output', 'artifacts/baseline.json')
run(PYTHON, '-m', 'ticket_router.evaluate', 'run', '--model', 'artifacts/phi4-router-merged', '--output', 'artifacts/tuned.json')
run(PYTHON, '-m', 'ticket_router.evaluate', 'compare', 'artifacts/baseline.json', 'artifacts/tuned.json', '--output', 'artifacts/comparison.json')
run(PYTHON, '-m', 'scripts.workflow', 'environment', '--stage', 'final')


## Read the measured results
Charts derive from the actual prediction reports. Completion means the workflow executed; it does not mean the candidate improved.


In [ ]:
python_cell("""
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
for name in ['baseline', 'tuned']:
    report = json.loads(Path(f'artifacts/{name}.json').read_text())
    print(name, json.dumps(report['metrics'], indent=2))
    cm = report['metrics']['confusion_matrix']
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.imshow(cm['values'], cmap='Blues')
    ax.set_xticks(range(len(cm['columns'])), cm['columns'], rotation=45, ha='right')
    ax.set_yticks(range(len(cm['rows'])), cm['rows'])
    ax.set(xlabel='Predicted', ylabel='True', title=name)
    for i, row in enumerate(cm['values']):
        for j, value in enumerate(row): ax.text(j, i, str(value), ha='center', va='center')
    fig.tight_layout()
    fig.savefig(f'artifacts/{name}-confusion.png')
    plt.close(fig)
""")
for name in ['baseline', 'tuned']:
    display(Image(filename=f'artifacts/{name}-confusion.png'))
display(json.loads(Path('artifacts/comparison.json').read_text()))
run(PYTHON, '-m', 'examples.week4_export')


## Download evidence — also run this after a failed cell
Includes actual stage logs, environment snapshots, exact source, configurations, prepared dataset and tokenizer templates.
The archive includes ticket text: keep it private unless you have permission to redistribute the data.
Download the **executed notebook** separately using Colab's File menu; the source notebook inside the ZIP has no execution output.
The evidence export does not label a failed run successful or fabricate missing reports.


In [ ]:
# Evidence packaging needs only the standard library, even if installation failed.
run(sys.executable, '-m', 'scripts.workflow', 'evidence')
files.download(str(root / 'week5-evidence.zip'))


## Save merged model weights
Do this before Colab expires if you need local inference. The evidence ZIP does not contain model weights.


In [ ]:
if input('Download merged model weights now? [y/N] ').strip().lower() == 'y':
    shutil.make_archive(str(root / 'phi4-router-merged'), 'zip', 'artifacts/phi4-router-merged')
    files.download(str(root / 'phi4-router-merged.zip'))
